# 🛎️ RecallDB — U.S. Product Recall Sample

A normalized, **provenance-tracked** sample of official U.S. federal product recalls, joining five source families into one model: **CPSC** (consumer products), **NHTSA** (vehicles), **FDA** (food/drug/device + safety alerts), **FSIS** (meat/poultry/egg), and **USCG** (recreational boating).

This public sample holds **200 recalls** (40 per agency) with the same table shapes as the paid full dataset (**127,783 recalls · 292,790 products**). Full dataset: [recalldb-public.pages.dev](https://recalldb-public.pages.dev/).


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

# Discover the sample regardless of the exact mount path / subfolder
BASE = "."
for root, _dirs, files in os.walk("/kaggle/input"):
    if "recalls.csv" in files:
        BASE = root
        break
print("Data directory:", BASE)

recalls   = pd.read_csv(f"{BASE}/recalls.csv")
products  = pd.read_csv(f"{BASE}/recalled_products.csv")
sources   = pd.read_csv(f"{BASE}/data_sources.csv")
firms     = pd.read_csv(f"{BASE}/firms.csv")
hazards   = pd.read_csv(f"{BASE}/hazards.csv")

for name, df in [("recalls", recalls), ("recalled_products", products),
                 ("data_sources", sources), ("firms", firms), ("hazards", hazards)]:
    print(f"{name:>18}: {df.shape[0]:>4} rows x {df.shape[1]} cols")
recalls.head()

## Recalls by federal agency

The sample is balanced across all five official source agencies (40 each).

In [ ]:
ax = (recalls['source_agency'].value_counts()
      .sort_values().plot.barh(figsize=(8,3.5), color='#14b8a6'))
ax.set_title('Sample recalls by source agency'); ax.set_xlabel('recalls')
plt.tight_layout(); plt.show()

## Hazard taxonomy

`hazards.csv` maps agency-specific free text into one deterministic vocabulary. `recall_count` reflects the **full dataset** distribution.

In [ ]:
ax = (hazards.sort_values('recall_count')
      .plot.barh(x='description', y='recall_count', legend=False,
                 figsize=(8,4.5), color='#f97316'))
ax.set_title('Full-dataset recalls per hazard category'); ax.set_xlabel('recalls'); ax.set_ylabel('')
plt.tight_layout(); plt.show()

## Provenance — every recall joins back to its source pull

`source_id` links each recall to `data_sources.csv`: the endpoint URL, retrieval timestamp, raw payload size and a SHA-256 fingerprint of the archived payload.

In [ ]:
prov = recalls.merge(sources, on='source_id', suffixes=('', '_src'))
prov[['recall_id','source_agency','recall_date','endpoint_url',
      'retrieved_at','raw_payload_bytes','raw_payload_sha256']].head()

## Recall volume over time

Parse `recall_date` and count recalls per month across the sample window.

In [ ]:
r = recalls.copy()
r['recall_date'] = pd.to_datetime(r['recall_date'], errors='coerce')
by_month = r.dropna(subset=['recall_date']).groupby(r['recall_date'].dt.to_period('M')).size()
ax = by_month.plot(figsize=(9,3.5), marker='o', color='#0ea5e9')
ax.set_title('Sample recalls per month'); ax.set_xlabel(''); ax.set_ylabel('recalls')
plt.tight_layout(); plt.show()

## Recalled products & categories

`recalled_products.csv` links products/vehicles/lots to each recall — join on `recall_id`.

In [ ]:
top_cats = products['category'].fillna('(unspecified)').value_counts().head(10)
ax = top_cats.sort_values().plot.barh(figsize=(8,4), color='#8b5cf6')
ax.set_title('Most common recalled-product categories (sample)'); ax.set_xlabel('products')
plt.tight_layout(); plt.show()

## Going further

- **Full dataset** (127,783 recalls · 292,790 products · full provenance ledger · CSV + SQLite): [recalldb-public.pages.dev](https://recalldb-public.pages.dev/)
- **Sources & data dictionary**: [SOURCES.md](https://recalldb-public.pages.dev/SOURCES.md) · [DATA_DICTIONARY.md](https://recalldb-public.pages.dev/DATA_DICTIONARY.md)
- **License**: public sample is **CC0-1.0** (U.S. federal government publications).
